# Quantum Mean Estimation (QME)

What is QME?: It's a method that leverages the ability of quantum computers to perform additions over an exponentially large (2^n) state space.

The role of the anchor qubit: While the first n qubits act as the index of the integration lattice, the n+1 qubit (anchor) acts as a "memory" that holds the information of the function f(x).

Efficiency: The main advantage is that, by performing measurements on the anchor qubit, we are extracting global information about the function without having to explicitly calculate each point sequentially in classical memory.

The following is the complete implementation of the QME algorithm in PennyLane to calculate the integral of an arbitrary function f(x) = x^p on an interval [a, b]

### Step 1: Environment Setup and Normalization

QME requires that the function be normalized, since quantum circuits naturally operate with probabilities (amplitudes).

In [1]:
import pennylane as qml
import numpy as np
from scipy.integrate import quad

# Parámetros del Algoritmo
n_qubits = 8
N = 2**n_qubits
p, a, b = 4, -1.0, 1.0

# Discretización y normalización del integrando
x_values = np.linspace(a, b, N)
f_values = x_values**p
f_max = np.max(np.abs(f_values))
f_norm = f_values / f_max # Esencial para mapear a amplitudes

dev = qml.device("default.qubit", wires=n_qubits+1)

### Step 2: State Preparation (Superposition)

Here, the index qubits are placed in a uniform superposition. This creates the "space" where we will calculate each point of the mesh simultaneously.
Python

In [7]:
@qml.qnode(dev)
def qme_circuit():
    # Inicialización: Hadamard a todos los qubits de índice
    for i in range(n_qubits):
        qml.Hadamard(wires=i)
    '''
    Integrand (Amplitude) Encoding: This is the critical step.
    We use RY rotations to "hide" the function value in the amplitude of the anchor qubit.
    The angle theta is adjusted so that the probability of measuring the anchor qubit is proportional to f(x).
    '''
    # Mapeo de valores a rotaciones RY controladas
    for i in range(n_qubits):
        for k in range(2**i):
            idx = k * 2**(n_qubits - i)
            if idx < len(f_norm):
                theta = 2 * np.arcsin(f_norm[idx])
                qml.RY(theta, wires=n_qubits)
    ''' 
    Measuring the Expected Value
    By measuring PauliZ on the anchor qubit, we obtain the expected value of the system, 
    which mathematically corresponds to the weighted sum (integral) we are looking for.
    '''
    return qml.expval(qml.PauliZ(n_qubits))

### Step 3: Classical Calculation and Validation

Finally, we scale the quantum result by multiplying it by the normalization factor (f_max) and the interval (b-a).

In [8]:
# Ejecución
expectation_z = qme_circuit()

# Reescalado para obtener la integral
integral_qme = (b - a) * f_max * expectation_z / N

# Comparación con método clásico (Riemann) y Exacto
integral_exact, _ = quad(lambda x: x**p, a, b)
integral_riemann = np.sum(f_values) * (b - a) / N